# ⚒️ **Vietnamese Capitalization and Punctuation Recovery using Transformer Seq2Seq Model (BARTpho)**

### 📘 Samsung Innovation Campus - AI Course - Capsule Project

> A deep learning-based solution for recovering punctuation and capitalization in Vietnamese texts using BARTpho-syllable and BLEU score evaluation.

---

## 🎓 Group's name: **404 Not Found**
👨‍💼 **Team Leader:** SIC250127 - Hoàng Phi Hùng  
👥 **Team Members:**
- SIC250159 - Nguyễn Vũ Thiện Nhân  
- SIC250155 - Thái Lê Hùng  
- SIC250175 - Nguyễn Thị Anh Thư

---

## 📚 References
- Science Paper: https://arxiv.org/abs/2207.01312
- Dataset: https://huggingface.co/datasets/vietgpt/binhvq_news_vi
- Github: https://github.com/bmd1905/vietnamese-correction

---

## **⛅0. Mount Google Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **🧱1. Initialize the Environment**

In [ ]:
!pip install torch transformers transformers[torch] sentencepiece datasets evaluate scikit-learn gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

## **📚2. Import Libraries**

In [ ]:
import json
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import Dataset as HFDataset
import evaluate
from sklearn.model_selection import train_test_split
import re
import gradio as gr

## **📄3. Create/Load Data**

In [ ]:
with open("/content/drive/MyDrive/SIC2025 - 404 Not Found/Crawl Dataset/dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

original_texts = [sample["original_text"] for sample in data]
corrected_texts = [sample["corrected_text"] for sample in data]

train_texts, test_texts, train_labels, test_labels = train_test_split(
    original_texts,
    corrected_texts,
    test_size=0.2,
    random_state=42
)

## **⚙️4. Preprocess Data**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("vinai/bartpho-syllable")

def preprocess_function(example):
    model_inputs = tokenizer(example["input"], padding="max_length", truncation=True, max_length=256)
    labels = tokenizer(example["output"], padding="max_length", truncation=True, max_length=256)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
train_data = [{"input": inp, "output": out} for inp, out in zip(train_texts, train_labels)]
test_data = [{"input": inp, "output": out} for inp, out in zip(test_texts, test_labels)]


In [ ]:
train_dataset = HFDataset.from_list(train_data)
test_dataset = HFDataset.from_list(test_data)

In [ ]:
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

## **🧠5. Build Transformer Seq2Seq Model using BARTpho**

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("vinai/bartpho-syllable")

## **🏋️6. Train the Model**

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=7,
    per_device_eval_batch_size=7,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_dir='./logs',
    logging_steps=100,
    gradient_accumulation_steps=1,
    fp16=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
)

trainer.train()

/tmp/ipython-input-11-633034059.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss
1,0.039900,0.035333
2,0.025300,0.029943
3,0.014500,0.027479


TrainOutput(global_step=3429, training_loss=0.034017076694642046, metrics={'train_runtime': 2459.2998, 'train_samples_per_second': 9.759, 'train_steps_per_second': 1.394, 'total_flos': 1.3002778607616e+16, 'train_loss': 0.034017076694642046, 'epoch': 3.0})

## **📈7. Evaluate the Model**

In [ ]:
bleu = evaluate.load("bleu")

def compute_bleu(model, tokenizer, test_texts, test_labels):
    predictions = []

    for inp in test_texts:
        input_ids = tokenizer.encode(inp, return_tensors="pt", truncation=True).to(model.device)
        output_ids = model.generate(input_ids, num_beams=4)
        output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        predictions.append(output_text)

    bleu_score = bleu.compute(predictions=predictions, references=[[label] for label in test_labels])
    return bleu_score


In [ ]:
result = compute_bleu(model, tokenizer, test_texts, test_labels)

In [ ]:
print("BLEU score:", result["bleu"])

BLEU score: 0.3059390516051032


In [ ]:
def compute_token_accuracy(model, tokenizer, test_texts, test_labels):
    correct_tokens = 0
    total_tokens = 0

    for inp, label in zip(test_texts, test_labels):
        input_ids = tokenizer.encode(inp, return_tensors="pt", truncation=True).to(model.device)

        attention_mask = (input_ids != tokenizer.pad_token_id).long().to(model.device)

        output_ids = model.generate(input_ids, attention_mask=attention_mask, num_beams=4, max_length=256)
        output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        predicted_tokens = tokenizer.tokenize(output_text)
        true_tokens = tokenizer.tokenize(label)

        min_len = min(len(predicted_tokens), len(true_tokens))
        for i in range(min_len):
            if predicted_tokens[i] == true_tokens[i]:
                correct_tokens += 1

        total_tokens += len(true_tokens)

    if total_tokens == 0:
        return 0.0

    accuracy = correct_tokens / total_tokens
    return accuracy

token_accuracy = compute_token_accuracy(model, tokenizer, test_texts, test_labels)
print("Token-level Accuracy:", token_accuracy)

Token-level Accuracy: 0.5417244247296923


## **👾8. Export the Model**

In [ ]:
output_dir = "/content/drive/MyDrive/SIC2025 - 404 Not Found/Model"

import os
os.makedirs(output_dir, exist_ok=True)

model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('/content/drive/MyDrive/SIC2025 - 404 Not Found/Model/tokenizer_config.json',
 '/content/drive/MyDrive/SIC2025 - 404 Not Found/Model/special_tokens_map.json',
 '/content/drive/MyDrive/SIC2025 - 404 Not Found/Model/sentencepiece.bpe.model',
 '/content/drive/MyDrive/SIC2025 - 404 Not Found/Model/dict.txt',
 '/content/drive/MyDrive/SIC2025 - 404 Not Found/Model/added_tokens.json')

## **🤖9. Use the Model**

In [2]:
import re
def fix_spacing(text):
    text = re.sub(r'\s+([.,!?:;])', r'\1', text)
    text = re.sub(r'([.,!?:;])(?=\S)', r'\1 ', text)
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r'\"\s+', '"', text)
    text = re.sub(r'\s+\"', '"', text)
    return text.strip()

def separate_words_and_punctuations(text):
    tokens = re.findall(r"\w+|[.,;:?!…'\"]+", text, flags=re.UNICODE)
    return " ".join(tokens)

def correct_punctuation(text):
    text = separate_words_and_punctuations(text)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    outputs = model.generate(**inputs, max_length=128, num_beams=4)
    return fix_spacing(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_path = "/content/drive/MyDrive/SIC2025 - 404 Not Found/Model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

## **🌐10. Test the Model – Web Interface with Gradio**

In [4]:
import gradio as gr
interface = gr.Interface(fn=correct_punctuation,
                         inputs="text",
                         outputs="text",
                         title="Vietnamese Capitalization and Punctuation Recovery using Transformer Seq2Seq Model (BARTpho)",
                         description="Nhập câu để hệ thống sửa.")

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3882e43782b3aad50d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
